# ClarifySQL · Notebook 02 — The frozen SQL writer (baseline B1 and the oracle)

**Goal.** Measure how well the base model, with no adapter and no questions, turns a fully specified
request into SQL. We use the original Spider questions of our validation (537) and test (1,034)
databases. On these clear questions B1 and the oracle receive exactly the same input. The result is the
**upper bound** every later system is compared with, plus the real cost per query.

Each question gets one answer, and up to **2 repairs** if SQLite reports an error. The files keep the
answer *before* repair as well, so one run gives both B1 (no repair) and B2 (with repair).

**Time:** about 45 minutes. **GPU quota:** about 45 minutes.
**Settings:** Kaggle → *Session options* → **GPU T4 x2**, **Internet On**.
Colab → *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi

## 1. Get the project code

This cell clones the project from GitHub (branch `main`) into the notebook's working folder and makes
the `clarifysql` package importable. Running it again later just pulls the newest commits.

*Private repository?* Add a secret called `GITHUB_TOKEN` (Kaggle: **Add-ons → Secrets**; Colab: the
**key icon** in the left bar) and set `USE_TOKEN = True`. Our repository is public, so this is not needed.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Sathviktr/LMA_ClarifySQL.git"
BRANCH = "main"
USE_TOKEN = False  # only needed for a private repository

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/LMA_ClarifySQL"

if USE_TOKEN:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

sys.path.insert(0, f"{REPO}/src")
print("Running on:", "Kaggle" if ON_KAGGLE else "Colab / other", "| repo at", REPO)

In [ ]:
# Libraries that are not preinstalled (or too old) on Kaggle/Colab. PyTorch with CUDA is already there.
!pip install -q -r {REPO}/requirements-gpu.txt
!pip install -q -e {REPO} --no-deps  # lets the scripts import clarifysql

## 2. Download Spider (about 1 minute)

The data goes to `/tmp/clarifysql_data`, which is outside the notebook's output folder, so the saved
notebook version stays small. The script compares the archive's SHA-256 fingerprint with the one in the
repository's `data/manifest.json` and says whether they match.

*If Google Drive refuses ("too many users"):* upload `spider_data.zip` from your laptop
(`C:\Users\sathv\clarifysql_data\downloads`) as a private Kaggle Dataset, attach it with
**Add Input**, and copy it to `/tmp/clarifysql_data/downloads/` before running the cell again.

In [ ]:
os.environ["CLARIFYSQL_DATA"] = "/tmp/clarifysql_data"
!python {REPO}/scripts/download_data.py --only spider

## 3. Check the pipeline without the model (seconds)

A fake "model" that answers with the gold SQL must score 100%. This checks the prompts, the SQL
extraction, the safe executor and the result comparison before we spend GPU time.

In [ ]:
!python {REPO}/scripts/run_direct.py --split validation --backend mock-gold --limit 50
!rm -f {REPO}/results/direct/*mock-gold*

## 4. Validation questions (537, about 10 minutes)

The model is loaded in FP16, and questions are sorted by length and sent in batches of 16 (the pilot
showed batching multiplies throughput). Greedy decoding throughout.

In [ ]:
!python {REPO}/scripts/run_direct.py --split validation --max-repairs 2 --batch-size 16

## 5. Test questions (1,034, about 20 minutes)

In [ ]:
!python {REPO}/scripts/run_direct.py --split test --max-repairs 2 --batch-size 16

## 6. Official test-suite accuracy (CPU, about 10 minutes)

Downloads the database variants of the test-suite evaluator (1.3 GB) and re-scores every test
prediction on all variants of its database. Set `RUN_TESTSUITE = False` to skip this; it can also be
done later on the laptop.

In [ ]:
RUN_TESTSUITE = True
if RUN_TESTSUITE:
    !python {REPO}/scripts/download_data.py --only testsuite
    !python {REPO}/scripts/score_testsuite.py {REPO}/results/direct/test_oracle_repair2.jsonl

## 7. Summary

In [ ]:
import json, glob
for path in sorted(glob.glob(f"{REPO}/results/direct/*_summary.json")):
    s = json.load(open(path))
    print(f"{s['split']:<10} items={s['items']}  accuracy before repair={s['execution_accuracy_before_repair']}"
          f"  after repair={s['execution_accuracy']}  test-suite={s.get('testsuite_accuracy', '-')}"
          f"  seconds/item={s['mean_seconds_per_item']}  wall={s['wall_seconds']}s")

## 8. Pack the results

In [ ]:
!cd {REPO} && zip -qr {WORK}/checkpoint2_results.zip results/direct
!ls -la {WORK}/checkpoint2_results.zip

## 9. Send the results back

Download `checkpoint2_results.zip`: on Kaggle from the **Output** panel (`/kaggle/working`), or on
Colab from the **folder icon** (`/content`). Unzip it **inside the repository folder** on your laptop,
so the files land in `LMA_ClarifySQL/results/direct/`. Then **stop the session** so it does not use up
GPU quota while idle.